In [394]:
import pandas as pd
import re
import os
import numpy as np
import requests
import matplotlib.pyplot as plt

In [395]:
os.getcwd()
#Num format
pd.options.display.float_format = '{:,.2f}'.format

path="scraping_output/final"

In [396]:
def clean_data(df):
    # Lowercase name and size
    df['name'] = df['name'].str.lower()
    df['size'] = df['size'].str.lower()

    # Infer type from name when type is NaN
    apt_kw  = r'departamento|depto|dpto|apto|apartamento|piso|flat|loft|studio|estudio'
    house_kw = r'\bcasa\b|chalet|villa|townhouse|bungalow'
    mask_nan = df['type'].isna()
    df.loc[mask_nan & df['name'].str.contains(apt_kw,   case=False, na=False), 'type'] = 'Apartment'
    df.loc[mask_nan & df['name'].str.contains(house_kw, case=False, na=False), 'type'] = 'House'

    #Keep apartments and houses
    df = df[df['type'].isin(['Apartment', 'House'])]

    def parse_number(val):
        """Parse number string handling dot/comma as thousand or decimal separator."""
        match = re.match(r'^[\d.,]+', str(val).strip())
        if not match:
            return np.nan
        s = match.group()

        has_dot = '.' in s
        has_comma = ',' in s

        if has_dot and has_comma:
            if s.rfind('.') > s.rfind(','):
                # e.g. 1,047.89 → comma=miles, dot=decimal
                s = s.replace(',', '')
            else:
                # e.g. 1.047,89 → dot=miles, comma=decimal
                s = s.replace('.', '').replace(',', '.')
        elif has_dot:
            # dot=thousands if ALL groups after dots have exactly 3 digits
            parts = s.split('.')
            if all(len(p) == 3 for p in parts[1:]):
                s = s.replace('.', '')
        elif has_comma:
            parts = s.split(',')
            if all(len(p) == 3 for p in parts[1:]):
                s = s.replace(',', '')
            else:
                s = s.replace(',', '.')

        return pd.to_numeric(s, errors='coerce')

    # Extract currency prefix from price string and update currency column.
    # Encuentra24 always uses $ = USD regardless of country; other sources use $ = local currency.
    def extract_currency(row):
        val = row['price']
        if pd.isna(val):
            return row['currency'], val
        s = str(val).strip()

        if row.get('source') == 'Encuentra24':
            # RD$5,500,000 → DOP (Dominican Peso)
            m_rd = re.match(r'^RD\$([\d.,].*)$', s)
            if m_rd:
                return 'DOP', m_rd.group(1)
            # Q850,000 → GTQ (Guatemalan Quetzal)
            m_q = re.match(r'^Q([\d.,].*)$', s)
            if m_q:
                return 'GTQ', m_q.group(1)
            # $185,000 or Desde$90,900 or $635,000-1% → USD
            m_usd = re.match(r'^[^$]*\$([\d.,].*)$', s)
            if m_usd:
                return 'USD', m_usd.group(1)

        # Standard letter-based prefix (e.g. USD, BRL, UF)
        m = re.match(r'^([A-Z]{2,4})\s*([\d.,].*)$', s)
        if m:
            detected = m.group(1)
            price_str = m.group(2)
            # Keep existing currency if already set, otherwise use detected one
            currency = row['currency'] if pd.notna(row['currency']) else detected
            return currency, price_str
        return row['currency'], val

    extracted = df[['price', 'currency', 'source']].apply(extract_currency, axis=1, result_type='expand')
    df['currency'] = extracted[0]
    df['price'] = extracted[1]

    # Convert price to numeric: remove "$" then parse
    def parse_price(val):
        if pd.isna(val):
            return np.nan
        return parse_number(str(val).replace('$', ''))

    # Convert size to numeric: remove "m²"/"m2" then parse
    def parse_size(val):
        if pd.isna(val):
            return np.nan
        return parse_number(str(val).replace('m²', '').replace('m2', ''))

    df['price'] = df['price'].apply(parse_price)
    df['size'] = df['size'].apply(parse_size)

    # Si el país es Uruguay, Paraguay o Bolivia, renombra price_usd a price y m2 a size
    special_countries = {"Uruguay", "Paraguay", "Bolivia"}
    df["price"] = df.apply(
        lambda row: row["price_usd"] if row["country"] in special_countries and pd.notna(row["price_usd"]) else row["price"],
        axis=1,
    )
    df["size"] = df.apply(
        lambda row: row["m2"] if row["country"] in special_countries and pd.notna(row["m2"]) else row["size"],
        axis=1,
    )
    # Añadir source "Infocasas" si país es Uruguay, Paraguay o Bolivia
    df["source"] = df.apply(
        lambda row: "Infocasas" if row["country"] in special_countries else row["source"],
        axis=1,
    )
    # Drop rows where price or size is NaN
    df = df.dropna(subset=['price', 'size'])
    #Dejar primera coincidencia de nombre y precio
    df = df.drop_duplicates(subset=['name', 'price'], keep='first')

    return df

In [397]:
##Load raw data

raw=pd.read_csv("scraping_output/raw/total_scraping.csv", low_memory=False)
print("Total rows", len(raw))
raw.head()

Total rows 99381


,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,region,locality,consultation_date,source,country,operation,price_usd,rooms,m2
0,Departamento en Venta en La Carolina,$ 140,NaN,Apartment,70 m²,1.0,2.0,-0.19,-78.49,"Eloy Alfaro Y de la Republica, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
1,Departamento en Venta en San Isidro del Inca,$ 69.900,NaN,Apartment,69 m²,3.0,2.0,-0.14,-78.46,"Calle N50G, Quito, ECU",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
2,Departamento en Venta en Centro De Quito,$ 68.000,NaN,Apartment,106 m²,3.0,2.0,-0.21,-78.50,"Asunción & Venezuela, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
3,Oficina en Venta en La Mariscal,$ 140.000,NaN,Accommodation,166 m²,7.0,2.0,-0.20,-78.49,Av. Francisco de Orellana & Avenida 10 de Agos...,Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
4,Departamento en Venta en Nayón,$ 110.000,NaN,Apartment,115 m²,2.0,3.0,-0.16,-78.44,"Jaime Roldos Aguilera & 19 de Diciembre, Quito...",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN


In [398]:
raw["currency"].unique()

array([nan, 'BRL'], dtype=object)

In [399]:
print("Data before cleaning")
print(raw.groupby(['country', 'operation']).size().unstack(fill_value=0).to_string())

Data before cleaning
operation    rent  sell
country                
Argentina    3000  3000
Bolivia      2100  2100
Brazil        844   812
Chile        6000  6000
Colombia     3000  3000
Costa-rica   3849  4000
Dominicana   1667  3689
Ecuador      3000  3000
El-salvador   666  1321
Guatemala    2764  4000
Honduras     1213  1652
Mexico       5162  7766
Nicaragua    1807  2142
Panama       3427  4000
Paraguay     2100  2100
Peru         3000  3000
Uruguay      2100  2100


In [401]:
## Clean data
clean_scrap=clean_data(raw.copy())
print("Total rows:", len(clean_scrap))
clean_scrap

Total rows: 55567


,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,region,locality,consultation_date,source,country,operation,price_usd,rooms,m2
0,departamento en venta en la carolina,140.00,NaN,Apartment,70.00,1.0,2.0,-0.19,-78.49,"Eloy Alfaro Y de la Republica, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
1,departamento en venta en san isidro del inca,"69,900.00",NaN,Apartment,69.00,3.0,2.0,-0.14,-78.46,"Calle N50G, Quito, ECU",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
2,departamento en venta en centro de quito,"68,000.00",NaN,Apartment,106.00,3.0,2.0,-0.21,-78.50,"Asunción & Venezuela, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
4,departamento en venta en nayón,"110,000.00",NaN,Apartment,115.00,2.0,3.0,-0.16,-78.44,"Jaime Roldos Aguilera & 19 de Diciembre, Quito...",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
5,casa en venta en garzota,"350,000.00",NaN,House,622.00,15.0,9.0,-2.15,-79.89,"Ciudadela Ietel, IETEL, Guayaquil, Ecuador",Guayas,Guayaquil,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99373,"arriendo parcela condominio carampangue, talag...","950,000.00",NaN,House,150.00,4.0,2.0,NaN,NaN,NaN,NaN,Talagante,2026-05-10 14:12:25,Yapo,Chile,rent,NaN,NaN,NaN
99375,exclusiva casa comercial a pasos de metro bilbao,"3,400,000.00",NaN,House,420.00,6.0,5.0,NaN,NaN,NaN,NaN,La Reina,2026-05-10 14:12:25,Yapo,Chile,rent,NaN,NaN,NaN
99377,cabaña alpina,"1,100,000.00",NaN,House,87.00,3.0,2.0,NaN,NaN,NaN,NaN,Pucón,2026-05-10 14:12:25,Yapo,Chile,rent,NaN,NaN,NaN
99378,casa en arriendo ojos del salado talagante,"550,000.00",NaN,House,80.00,3.0,2.0,NaN,NaN,NaN,NaN,Talagante,2026-05-10 14:12:25,Yapo,Chile,rent,NaN,NaN,NaN


In [404]:
# currency_map = {
#     'Argentina': 'ARS', 'Brazil': 'BRL', 'Chile': 'CLP',
#     'Colombia': 'COP', 'Costa-rica': 'CRC', 'Ecuador': 'USD',
#     'El-salvador': 'USD', 'Guatemala': 'GTQ', 'Honduras': 'HNL',
#     'Mexico': 'MXN', 'Nicaragua': 'NIO', 'Panama': 'USD', 'Peru': 'PEN'
# }

In [405]:
# # Get exchange rates (exchangerate-api)
# API_KEY="cf53b6293b066860dd66b072"
# url = f"https://v6.exchangerate-api.com/v6/{API_KEY}/latest/USD"
# r = requests.get(url).json()

# if r.get('result') == 'success':
#     rates = r['conversion_rates']
#     last_update = pd.to_datetime(r['time_last_update_utc']).strftime('%Y-%m-%d')

#     exchange_table = pd.DataFrame([
#         {
#             'country': country,
#             'rate': 1.0 if code == 'USD' else (1 / rates[code] if code in rates else None),
#             'last_update': last_update
#         }
#         for country, code in currency_map.items()
#     ])

#     print(exchange_table.to_string(index=False))
# else:
#     print("⚠️ Error al obtener tasas:", r)


# def get_latest_exchange_rate(country_code):
#     """
#     Devuelve el último tipo de cambio disponible (moneda local por 1 USD)
#     usando World Bank API (indicador PA.NUS.FCRF).

#     Parámetros
#     ----------
#     country_code : str  — ISO2 (e.g. 'MX', 'BR', 'CL')

#     Retorna
#     -------
#     dict con keys: country_code, source, date, exchange_rate
#     """
#     country_code = country_code.upper()
#     url = (
#         f"https://api.worldbank.org/v2/country/{country_code}"
#         f"/indicator/PA.NUS.FCRF"
#         f"?format=json&per_page=10&mrv=1"
#     )
#     r = requests.get(url)
#     r.raise_for_status()
#     records = r.json()[1]
#     if not records:
#         raise ValueError(f"Sin datos World Bank para {country_code}")
#     item = records[0]
#     return {
#         "country_code": country_code,
#         "source": "worldbank",
#         "date": item["date"],
#         "exchange_rate": item["value"],
#     }


# # ISO2 codes per country (USD countries get rate=1.0 directly)
# iso2_map = {
#     'Argentina':    'AR',
#     'Brazil':       'BR',
#     'Chile':        'CL',
#     'Colombia':     'CO',
#     'Costa-rica':   'CR',
#     'Ecuador':      'EC',
#     'El-salvador':  'SV',
#     'Guatemala':    'GT',
#     'Honduras':     'HN',
#     'Mexico':       'MX',
#     'Nicaragua':    'NI',
#     'Panama':       'PA',
#     'Peru':         'PE',
# }

# rows = []
# for country, iso2 in iso2_map.items():
#     if iso2 == 'USD':
#         rows.append({
#             'country':              country,
#             'lcu_per_usd':          1.0,    # local currency units per 1 USD (World Bank raw)
#             'rate':                 1.0,    # USD per 1 local unit (used for price conversion)
#             'last_update':          None,
#         })
#     else:
#         wb = get_latest_exchange_rate(iso2)
#         rows.append({
#             'country':              country,
#             'lcu_per_usd':          wb['exchange_rate'],          # World Bank raw value
#             'rate':                 1 / wb['exchange_rate'],      # USD per local unit
#             'last_update':          wb['date'],
#         })

# exchange_table = pd.DataFrame(rows)
# exchange_table

In [406]:
# Get exchange rates from IMF CSV — indicator: US Dollar per domestic currency, Period average
# Fallback per country: Monthly → Quarterly → Annual

COUNTRY_TO_IMF = {
    'Argentina':   'Argentina',
    'Brazil':      'Brazil',
    'Chile':       'Chile',
    'Colombia':    'Colombia',
    'Costa-rica':  'Costa Rica',
    'Dominicana':  'Dominican Republic',
    'Guatemala':   'Guatemala',
    'Honduras':    'Honduras',
    'Mexico':      'Mexico',
    'Nicaragua':   'Nicaragua',
    'Peru':        'Peru',
    'Ecuador':     'Ecuador',
    'El-salvador': 'El Salvador',
    'Panama':      'Panama',
    'Uruguay':    'Uruguay',
    "Paraguay":   "Paraguay",
    "Bolivia":   "Bolivia"
}

imf_csv = pd.read_csv(
    "../exchange_rates_imf.csv",
    on_bad_lines="skip",
    engine="python",
    encoding="utf-8",
    usecols=["COUNTRY", "INDICATOR", "TYPE_OF_TRANSFORMATION", "FREQUENCY", "TIME_PERIOD", "OBS_VALUE"],
)

def parse_imf_period(val):
    if "-M" in str(val):
        return pd.to_datetime(val, format="%Y-M%m")
    if "-Q" in str(val):
        year, q = val.split("-Q")
        return pd.Timestamp(year=int(year), month=(int(q) - 1) * 3 + 1, day=1)
    try:
        return pd.Timestamp(year=int(val), month=1, day=1)
    except Exception:
        return pd.NaT

imf_base = (
    imf_csv
    .loc[
        (imf_csv["INDICATOR"] == "US Dollar per domestic currency") &
        (imf_csv["TYPE_OF_TRANSFORMATION"] == "Period average") &
        (imf_csv["COUNTRY"].isin(COUNTRY_TO_IMF.values()))
    ]
    .assign(
        period=lambda d: d["TIME_PERIOD"].map(parse_imf_period),
        rate=lambda d: pd.to_numeric(d["OBS_VALUE"], errors="coerce"),
    )
    .dropna(subset=["rate", "period"])
)

def last_by_freq(freq):
    sub = imf_base[imf_base["FREQUENCY"] == freq]
    if sub.empty:
        return {}
    return (
        sub.loc[sub.groupby("COUNTRY")["period"].idxmax()]
           .set_index("COUNTRY")[["rate", "period"]]
           .to_dict("index")
    )

freq_data = {f: last_by_freq(f) for f in ["Monthly", "Quarterly", "Annual"]}

rows = []
for country, imf_name in COUNTRY_TO_IMF.items():
    for freq in ["Monthly", "Quarterly", "Annual"]:
        if imf_name in freq_data[freq]:
            rec = freq_data[freq][imf_name]
            rows.append({
                "country":     country,
                "rate":        rec["rate"],
                "last_update": rec["period"].strftime("%Y-%m"),
            })
            break

exchange_table = pd.DataFrame(rows).sort_values("country").reset_index(drop=True)
exchange_table.style.format({"rate": "{:.6f}"})

,country,rate,last_update
0,Argentina,0.000806,2025-01
1,Bolivia,0.144718,2026-01
2,Brazil,0.178978,2025-01
3,Chile,0.001051,2025-01
4,Colombia,0.000264,2025-12
5,Costa-rica,0.002131,2026-03
6,Dominicana,0.015746,2026-01
7,Ecuador,1.000000,2026-03
8,El-salvador,1.000000,2026-03
9,Guatemala,0.130646,2026-03


In [407]:
clean_scrap = clean_scrap.merge(
    exchange_table[['country', 'rate', 'last_update']],
    on='country',
    how='left'
)

# Create price_usd:
# - currency == 'USD' → direct price
# - currency == 'UF'  → price * uf_value (CLP) * rate_to_usd del país (CLP→USD)
# - NaN o moneda local → price * rate_to_usd
clean_scrap['price_usd'] = clean_scrap.apply(
    lambda row: row['price'] if row['currency'] == 'USD' or row['country'] in {'Uruguay', 'Paraguay', 'Bolivia'}
    #UF from Banco Central de Chile
    #https://tinyurl.com/yz5ffy3n
                else row['price'] *39841.72* row['rate'] if row['currency'] == 'UF'
                else row['price'] * row['rate'],
    axis=1
)

#Price per square meter
clean_scrap['price_per_sq_meter'] = clean_scrap['price_usd'] / clean_scrap['size']

# Absolute filters — rent and sell have very different price_per_sq_meter scales
rent_mask = clean_scrap['operation'] == 'rent'
sell_mask = clean_scrap['operation'] == 'sell'

clean_scrap = clean_scrap[
    (clean_scrap['size'] >= 15) &
    (clean_scrap['size'] <= 600) &
    (clean_scrap['price_usd'] >= 50) &
    (clean_scrap['price_per_sq_meter'] >= 0.5) &
    (
        (rent_mask & (clean_scrap['price_per_sq_meter'] <= 300)) |
        (sell_mask & (clean_scrap['price_per_sq_meter'] <= 15000))
    )
]

clean_scrap

,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,...,consultation_date,source,country,operation,price_usd,rooms,m2,rate,last_update,price_per_sq_meter
0,departamento en venta en la carolina,140.00,NaN,Apartment,70.00,1.0,2.0,-0.19,-78.49,"Eloy Alfaro Y de la Republica, Quito, Ecuador",...,2026-03-19 18:26:04,Properati,Ecuador,sell,140.00,NaN,NaN,1.00,2026-03,2.00
1,departamento en venta en san isidro del inca,"69,900.00",NaN,Apartment,69.00,3.0,2.0,-0.14,-78.46,"Calle N50G, Quito, ECU",...,2026-03-19 18:26:04,Properati,Ecuador,sell,"69,900.00",NaN,NaN,1.00,2026-03,"1,013.04"
2,departamento en venta en centro de quito,"68,000.00",NaN,Apartment,106.00,3.0,2.0,-0.21,-78.50,"Asunción & Venezuela, Quito, Ecuador",...,2026-03-19 18:26:04,Properati,Ecuador,sell,"68,000.00",NaN,NaN,1.00,2026-03,641.51
3,departamento en venta en nayón,"110,000.00",NaN,Apartment,115.00,2.0,3.0,-0.16,-78.44,"Jaime Roldos Aguilera & 19 de Diciembre, Quito...",...,2026-03-19 18:26:04,Properati,Ecuador,sell,"110,000.00",NaN,NaN,1.00,2026-03,956.52
5,casa en venta en tingo,"220,000.00",NaN,House,407.00,3.0,3.0,-0.29,-78.44,"Ilaló &, Quito, Ecuador",...,2026-03-19 18:26:04,Properati,Ecuador,sell,"220,000.00",NaN,NaN,1.00,2026-03,540.54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55562,"arriendo parcela condominio carampangue, talag...","950,000.00",NaN,House,150.00,4.0,2.0,NaN,NaN,NaN,...,2026-05-10 14:12:25,Yapo,Chile,rent,998.60,NaN,NaN,0.00,2025-01,6.66
55563,exclusiva casa comercial a pasos de metro bilbao,"3,400,000.00",NaN,House,420.00,6.0,5.0,NaN,NaN,NaN,...,2026-05-10 14:12:25,Yapo,Chile,rent,"3,573.95",NaN,NaN,0.00,2025-01,8.51
55564,cabaña alpina,"1,100,000.00",NaN,House,87.00,3.0,2.0,NaN,NaN,NaN,...,2026-05-10 14:12:25,Yapo,Chile,rent,"1,156.28",NaN,NaN,0.00,2025-01,13.29
55565,casa en arriendo ojos del salado talagante,"550,000.00",NaN,House,80.00,3.0,2.0,NaN,NaN,NaN,...,2026-05-10 14:12:25,Yapo,Chile,rent,578.14,NaN,NaN,0.00,2025-01,7.23


In [408]:
## Percentile filtering per group (country, type, operation)
# Removes values outside the 2nd–98th percentile within each group
# Applied to price_usd, size, and price_per_sq_meter independently

GROUP_COLS = ['country', 'type', 'operation']
LOW_PCT = 0.02
HIGH_PCT = 0.98
MIN_GROUP_SIZE = 10  # skip groups too small to compute percentiles reliably

def filter_percentile(df, col, group_cols, low=LOW_PCT, high=HIGH_PCT, min_size=MIN_GROUP_SIZE):
    mask = pd.Series(True, index=df.index)
    for _, group_idx in df.groupby(group_cols).groups.items():
        if len(group_idx) < min_size:
            continue
        vals = df.loc[group_idx, col]
        lo = vals.quantile(low)
        hi = vals.quantile(high)
        mask.loc[group_idx] = vals.between(lo, hi)
    return df[mask]

before = len(clean_scrap)
clean_scrap = filter_percentile(clean_scrap, 'price_usd', GROUP_COLS)
after_price = len(clean_scrap)
clean_scrap = filter_percentile(clean_scrap, 'size', GROUP_COLS)
after_size = len(clean_scrap)
clean_scrap = filter_percentile(clean_scrap, 'price_per_sq_meter', GROUP_COLS)
after_ppsm = len(clean_scrap)

# Recalculate price_per_sq_meter after filtering
clean_scrap['price_per_sq_meter'] = clean_scrap['price_usd'] / clean_scrap['size']

print(f"Rows before percentile filter       : {before:,}")
print(f"After price_usd filter              : {after_price:,}  (removed {before - after_price:,})")
print(f"After size filter                   : {after_size:,}   (removed {after_price - after_size:,})")
print(f"After price_per_sq_meter filter     : {after_ppsm:,}   (removed {after_size - after_ppsm:,})")
print(f"Total removed                       : {before - after_ppsm:,}  ({(before - after_ppsm) / before:.1%})")

Rows before percentile filter       : 48,328
After price_usd filter              : 46,423  (removed 1,905)
After size filter                   : 44,671   (removed 1,752)
After price_per_sq_meter filter     : 42,840   (removed 1,831)
Total removed                       : 5,488  (11.4%)


In [409]:
clean_scrap["currency"].unique()

array([nan, 'USD', 'GTQ', 'DOP', 'BRL', 'UF'], dtype=object)

In [410]:
clean_scrap["type"].unique()

array(['Apartment', 'House'], dtype=object)

In [411]:
print("\nData after cleaning")
print(clean_scrap.groupby(['country', 'type', 'operation']).size().unstack(fill_value=0).to_string())


Data after cleaning
operation              rent  sell
country     type                 
Argentina   Apartment   311   329
            House       192   329
Bolivia     Apartment   467   296
            House       737   651
Brazil      Apartment   628   526
            House       105   173
Chile       Apartment  1319  2264
            House      1477  2353
Colombia    Apartment   486   256
            House       238   365
Costa-rica  Apartment   744  1250
            House       622  1006
Dominicana  Apartment   294  1246
            House        17   714
Ecuador     Apartment   443   399
            House       457   820
El-salvador Apartment    94   336
            House        70   594
Guatemala   Apartment   294  1270
            House        82  1103
Honduras    Apartment    35   252
            House         6   380
Mexico      Apartment  1199   910
            House       891  3265
Nicaragua   Apartment    16    72
            House       183   836
Panama      Apartment  1220

In [412]:
print("\nsources")
print(clean_scrap.groupby('country')['source'].unique().to_string())


sources
country
Argentina            [Properati]
Bolivia              [Infocasas]
Brazil             [QuintoAndar]
Chile                     [Yapo]
Colombia             [Properati]
Costa-rica         [Encuentra24]
Dominicana         [Encuentra24]
Ecuador              [Properati]
El-salvador        [Encuentra24]
Guatemala          [Encuentra24]
Honduras           [Encuentra24]
Mexico         [Pincali, iCasas]
Nicaragua          [Encuentra24]
Panama             [Encuentra24]
Paraguay             [Infocasas]
Peru                 [Properati]
Uruguay              [Infocasas]


In [413]:
## Save clean_data
clean_scrap.to_csv(path + "/clean_scrap.csv",  index=False, encoding="utf-8-sig")

### Summary stats

In [414]:
summary_by_country_type_operation = clean_scrap.groupby(['country', 'type', 'operation']).agg(
    listings=('price', 'size'),
    avg_price_usd=('price_usd', 'mean'),
    med_price_usd=('price_usd', 'median'),
    min_price_usd=('price_usd', 'min'),
    max_price_usd=('price_usd', 'max'),
    avg_size=('size', 'mean'),
    med_size=('size', 'median'),
    min_size=('size', 'min'),
    max_size=('size', 'max'),
    avg_price_per_sq_meter=('price_per_sq_meter', 'mean'),
).sort_values(['country', 'listings'], ascending=[True, False])

summary_by_country_type_operation = summary_by_country_type_operation.round({
    'avg_price_usd': 2,
    'med_price_usd': 2,
    'min_price_usd': 2,
    'max_price_usd': 2,
    'avg_size': 2,
    'med_size': 2,
    'min_size': 2,
    'max_size': 2,
    'avg_price_per_sq_meter': 2,
})

for country, country_summary in summary_by_country_type_operation.groupby(level='country'):
    country_table = country_summary.reset_index(level='country', drop=True).reset_index()
    country_table['group'] = country_table['type'] + ' | ' + country_table['operation']
    country_table = country_table.set_index('group')[
        ['listings', 'avg_price_usd', 'med_price_usd', 'min_price_usd', 'max_price_usd',
         'avg_size', 'med_size', 'min_size', 'max_size', 'avg_price_per_sq_meter']
    ]

    print(f"\nSummary for {country}")
    print(country_table.to_string())

    fig, ax = plt.subplots(figsize=(18, max(4, 0.35 * len(country_table))))
    fig.suptitle(f"Summary statistics for {country} (grouped by type and operation)", fontsize=14, y=0.95)
    ax.axis('off')
    table = ax.table(
        cellText=country_table.reset_index().values,
        colLabels=['group', 'listings', 'avg_price', 'med_price', 'min_price', 'max_price',
                   'avg_size', 'med_size', 'min_size', 'max_size', 'avg_price_m2'],
        cellLoc='center',
        loc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.1)

    image_file = os.path.join(
        path,
        f"summary_{country.lower().replace(' ', '_').replace('-', '_')}.png",
    )
    plt.savefig(image_file, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved summary image for {country} to: {image_file}")



Summary for Argentina
                  listings  avg_price_usd  med_price_usd  min_price_usd  max_price_usd  avg_size  med_size  min_size  max_size  avg_price_per_sq_meter
group                                                                                                                                                 
Apartment | sell       329     133,482.73     110,000.00      27,000.00     450,000.00    111.72     67.00     25.00    500.00                1,604.26
House | sell           329     144,759.80     118,000.00      21,000.00     470,000.00    157.73    127.00     27.00    480.00                1,208.19
Apartment | rent       311         925.45         604.42          60.44       4,950.00     96.42     60.00     23.00    460.00                   10.73
House | rent           192       1,423.17         853.24          64.47       7,900.00    122.83    110.00     25.00    450.00                   12.28


Saved summary image for Argentina to: scraping_output/final\summary_argentina.png

Summary for Bolivia
                  listings  avg_price_usd  med_price_usd  min_price_usd  max_price_usd  avg_size  med_size  min_size  max_size  avg_price_per_sq_meter
group                                                                                                                                                 
House | rent           737       1,636.68       1,300.00         285.00       5,000.00    283.69    264.00     85.00    570.00                    5.58
House | sell           651     231,360.42     184,000.00      48,571.00     800,000.00    255.95    229.00     87.00    560.00                  880.54
Apartment | rent       467         774.20         600.00         215.00       3,600.00    100.04     85.00     30.00    350.00                    8.60
Apartment | sell       296     152,577.35     115,000.00      29,900.00     643,500.00    160.92    126.00     33.00    550.00                

In [415]:
## Presentation summary — median price, size, price/m² and listings by country, type, operation

metrics = (
    clean_scrap
    .groupby(['country', 'operation', 'type'])
    .agg(
        med_price = ('price_usd',           'median'),
        med_size  = ('size',                'median'),
        med_ppsm  = ('price_per_sq_meter',  'median'),
        n         = ('price_usd',           'count'),
    )
    .round({'med_price': 0, 'med_size': 0, 'med_ppsm': 0})
    .reset_index()
)

# One source string per country (joined if multiple)
source_map = (
    clean_scrap
    .groupby('country')['source']
    .agg(lambda x: ', '.join(sorted(x.unique())))
    .to_dict()
)

pivot = (
    metrics
    .pivot_table(index=['country', 'operation'], columns='type',
                 values=['med_price', 'med_size', 'med_ppsm', 'n'],
                 aggfunc='first')
)
pivot.columns = [f"{t} — {m}" for m, t in pivot.columns]
pivot = pivot.reset_index()

for op in ['rent', 'sell']:
    sub = pivot[pivot['operation'] == op].drop(columns='operation').set_index('country')
    col_order = (
        [c for c in sub.columns if 'Apartment' in c] +
        [c for c in sub.columns if 'House' in c]
    )
    sub = sub[col_order]
    sub.columns = pd.MultiIndex.from_tuples(
        [(c.split(' — ')[0], c.split(' — ')[1]) for c in sub.columns]
    )
    sub.index.name = None

    print(f"\n{'='*70}")
    print(f"  {'RENT' if op == 'rent' else 'SELL'}  |  Median price (USD)  ·  Median size (m²)  ·  Price/m²  ·  N listings")
    print(f"{'='*70}")

    fmt = {'med_price': '{:,.0f}', 'med_size': '{:,.0f}', 'med_ppsm': '{:,.0f}', 'n': '{:,.0f}'}
    styled = sub.style.format(
        {(t, m): fmt[m]
         for t in ['Apartment', 'House']
         for m in ['med_price', 'med_size', 'med_ppsm', 'n']
         if (t, m) in sub.columns}
    )
    display(styled)


  RENT  |  Median price (USD)  ·  Median size (m²)  ·  Price/m²  ·  N listings



  SELL  |  Median price (USD)  ·  Median size (m²)  ·  Price/m²  ·  N listings


In [416]:
## Save presentation summary as separate HTML files + PNG screenshots

METRIC_ORDER   = ['Median Price (USD)', 'Median Price / m² (USD)', 'Median Size (m²)', 'Listings']
RENAME_METRICS = {
    'med_price': 'Median Price (USD)',
    'med_size':  'Median Size (m²)',
    'med_ppsm':  'Median Price / m² (USD)',
    'n':         'Listings',
}

FONT = "'Century Gothic', CenturyGothic, AppleGothic, sans-serif"

def build_styled_table(pivot, op, source_map):
    sub = pivot[pivot['operation'] == op].drop(columns='operation').set_index('country')
    col_order = (
        [c for c in sub.columns if 'Apartment' in c] +
        [c for c in sub.columns if 'House' in c]
    )
    sub = sub[col_order]
    sub.columns = pd.MultiIndex.from_tuples(
        [(c.split(' — ')[0], RENAME_METRICS[c.split(' — ')[1]]) for c in sub.columns]
    )
    prop_types = sub.columns.get_level_values(0).unique()
    ordered_cols = [
        (pt, m)
        for pt in prop_types
        for m in METRIC_ORDER
        if (pt, m) in sub.columns
    ]
    sub = sub[ordered_cols]
    sub.index.name = None

    src = pd.DataFrame(
        {('', 'Source'): sub.index.map(source_map)},
        index=sub.index
    )
    sub = pd.concat([src, sub], axis=1)

    numeric_cols = [c for c in sub.columns if c[1] in METRIC_ORDER]
    fmt = {c: '{:,.0f}' for c in numeric_cols}

    styled = (
        sub.style
        .format(fmt)
        .set_table_styles([
            {'selector': 'thead th',
             'props': [('background-color', '#1a1a2e'), ('color', 'white'),
                       ('padding', '8px 14px'), ('text-align', 'center'),
                       ('border', '1px solid #444'), ('font-family', FONT)]},
            {'selector': 'tbody th',
             'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'),
                       ('padding', '7px 14px'), ('border', '1px solid #ddd'),
                       ('font-family', FONT)]},
            {'selector': 'td',
             'props': [('padding', '7px 14px'), ('text-align', 'right'),
                       ('border', '1px solid #ddd'), ('background-color', 'white'),
                       ('font-family', FONT)]},
            {'selector': 'tr:hover td',
             'props': [('background-color', '#fff9e6')]},
            {'selector': 'table',
             'props': [('border-collapse', 'collapse'), ('font-family', FONT),
                       ('font-size', '13px')]},
        ])
    )
    return styled.to_html()

HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <title>Real Estate LAC — {{label}}</title>
  <style>
    html, body {{
      overflow: hidden; margin: 0; padding: 0; background: #fafafa;
      font-family: {font};
    }}
    .wrapper {{ display: inline-block; padding: 48px; }}
    h1    {{ font-family: {font}; font-size: 20px; margin: 0 0 4px 0;
             color: #1a1a2e; font-weight: bold; }}
    p.sub {{ font-family: {font}; font-size: 13px; color: #555; margin: 0 0 28px 0; }}
  </style>
</head>
<body>
  <div class="wrapper">
    <h1>Real Estate — Latin America &amp; Caribbean &nbsp;|&nbsp; {{label}}</h1>
    <p class="sub">Median listing price and size by country and property type.</p>
    {{table}}
  </div>
</body>
</html>""".format(font=FONT)

# Save separate HTML files
html_paths = {}
for op in ['rent', 'sell']:
    label      = op.upper()
    table_html = build_styled_table(pivot, op, source_map)
    content    = HTML_TEMPLATE.replace('{label}', label).replace('{table}', table_html)
    fpath      = os.path.abspath(os.path.join(path, f"summary_{op}.html"))
    with open(fpath, 'w', encoding='utf-8') as f:
        f.write(content)
    html_paths[op] = fpath
    print(f"Saved HTML : {fpath}")

# Capture PNG screenshots with selenium + Pillow crop
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from PIL import Image
    import io, time

    opts = Options()
    opts.add_argument('--headless=new')
    opts.add_argument('--disable-gpu')
    opts.add_argument('--no-sandbox')

    driver = webdriver.Chrome(options=opts)

    for op, fpath in html_paths.items():
        driver.get(f'file:///{fpath}')
        driver.set_window_size(2400, 6000)
        time.sleep(0.5)

        info = driver.execute_script("""
            var el  = document.querySelector('.wrapper');
            var dpr = window.devicePixelRatio || 1;
            return { w: el.offsetWidth, h: el.offsetHeight, dpr: dpr };
        """)
        w_px = int(info['w'] * info['dpr'])
        h_px = int(info['h'] * info['dpr'])

        raw = driver.get_screenshot_as_png()
        img = Image.open(io.BytesIO(raw)).crop((0, 0, w_px, h_px))

        png_path = os.path.join(path, f"summary_{op}.png")
        img.save(png_path)
        print(f"Saved PNG  : {png_path}  ({w_px}×{h_px}px)")

    driver.quit()

except ImportError as e:
    print(f"⚠️  Falta dependencia: {e}  — ejecuta: pip install selenium pillow")
except Exception as e:
    print(f"⚠️  Error al capturar PNG: {e}")

Saved HTML : c:\Users\claud\Documents\GitHub\rent_lac\tests\scraping_output\final\summary_rent.html
Saved HTML : c:\Users\claud\Documents\GitHub\rent_lac\tests\scraping_output\final\summary_sell.html
Saved PNG  : scraping_output/final\summary_rent.png  (1658×913px)
Saved PNG  : scraping_output/final\summary_sell.png  (1658×913px)
